###Dimension Table (Device) Load 
1. CDC Bronze Curation Silver load and SCD1, SCD2, CDF Gold Load using Notebook using traditional merge statements

![](devices_cdc.png)

In [0]:
  -- Bronze Table
CREATE TABLE IF NOT EXISTS telecom_bronze.dim_device_bronze (
  device_id INT, device_type STRING, brand STRING, model STRING, os STRING, 
  owner_customer_id INT, status STRING, updated_at TIMESTAMP);
  -- Silver Table
CREATE TABLE IF NOT EXISTS telecom_silver.dim_device_silver (
    device_id STRING, brand STRING, model STRING, os STRING, 
    device_type STRING, owner_customer_id STRING, status STRING, updated_at TIMESTAMP
) USING DELTA;

-- Gold SCD1 Table (CDF Enabled)
CREATE TABLE IF NOT EXISTS telecom_gold.dim_device_gold_scd1 (
    device_id STRING, brand STRING, model STRING, os STRING, 
    device_type STRING, owner_customer_id STRING, status STRING, updated_at TIMESTAMP
) USING DELTA TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Gold SCD2 Table (Includes DLT-style history columns)
CREATE TABLE IF NOT EXISTS telecom_gold.dim_device_gold_scd2 (
    device_id STRING, brand STRING, model STRING, os STRING, 
    device_type STRING, owner_customer_id STRING, status STRING, updated_at TIMESTAMP,
    start_at TIMESTAMP, end_at TIMESTAMP
) USING DELTA;

In [0]:
select * from telecom_bronze.dim_device_bronze limit 10;

In [0]:
%sql
--1. CDC (History load for the first time alone) applying Watermarking Feature using Foreign Catalog
INSERT INTO telecom_bronze.dim_device_bronze
SELECT 
    device_id, 
    device_type, 
    brand, 
    model, 
    os, 
    owner_customer_id, 
    status, 
    updated_at
FROM telecom_foreign_catalog.telco_schema.dim_device
WHERE updated_at > (SELECT COALESCE(MAX(updated_at), '1900-01-01') 
    FROM telecom_bronze.dim_device_bronze);
    --This is checkpointing or watermarking feature to achieve incremental load/CDC

In [0]:
%python
from pyspark.sql.functions import col, upper, trim, when, row_number
from pyspark.sql.window import Window

# ====================================================================
# 1. READ BRONZE & LOAD CURATED SILVER (DEDUPLICATED BATCH)
# ====================================================================

raw_df = spark.read.table("telecom_bronze.dim_device_bronze")

# Define Window to identify the latest record per device in the incoming batch
window_spec = Window.partitionBy("device_id").orderBy(col("updated_at").desc())

# Apply filters, transformations, and deduplication
silver_df = (
    raw_df
    .filter(col("device_id").isNotNull()) 
    .withColumn("device_id", col("device_id").cast("string"))
    .withColumn("owner_customer_id", col("owner_customer_id").cast("string"))
    .withColumn("brand", upper(trim(col("brand"))))
    .withColumn("model", trim(col("model")))
    .withColumn("os", when(col("os").rlike("(?i)^ios$"), "iOS")
                      .otherwise(upper(trim(col("os")))))
    # Deduplication Logic
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Save the deduplicated batch to Silver
silver_df.write.format("delta").mode("append").saveAsTable("telecom_silver.dim_device_silver")

# ====================================================================
# 2. SHIFTING TO SQL MERGE FOR GOLD LAYER (SIMPLE UPSERT)
# ====================================================================

silver_df.createOrReplaceTempView("dedup_latest_silver")

# ====================================================================
# 3. GOLD LAYER: SCD TYPE 1 (Hard Deletes + Hash Comparison)
# ====================================================================
spark.sql("""
    MERGE INTO telecom_gold.dim_device_gold_scd1 t
    USING dedup_latest_silver s
    ON t.device_id = s.device_id
    
    WHEN MATCHED AND s.status = 'Inactive' THEN 
        DELETE -- Hard delete
        
    WHEN MATCHED AND s.updated_at > t.updated_at 
                 AND hash(s.device_type, s.brand, s.model, s.os, s.owner_customer_id, s.status) <> 
                     hash(t.device_type, t.brand, t.model, t.os, t.owner_customer_id, t.status) THEN 
        UPDATE SET *
        
    WHEN NOT MATCHED AND s.status != 'Inactive' THEN 
        INSERT *
""")

# ====================================================================
# 4. GOLD LAYER: SCD TYPE 2 (Soft Deletes + Hash Comparison)
# ====================================================================
staged_updates = spark.sql("""
    SELECT s.*, s.device_id as merge_key FROM dedup_latest_silver s--all source data
    UNION ALL
    SELECT s.*, NULL as merge_key FROM dedup_latest_silver s --existing active data in the target with source updated
    JOIN telecom_gold.dim_device_gold_scd2 t ON s.device_id = t.device_id
    WHERE t.end_at IS NULL 
      AND hash(s.device_type, s.brand, s.model, s.os, s.owner_customer_id, s.status) <> 
          hash(t.device_type, t.brand, t.model, t.os, t.owner_customer_id, t.status)""")

staged_updates.createOrReplaceTempView("staged_updates")

spark.sql("""
    MERGE INTO telecom_gold.dim_device_gold_scd2 t
    USING staged_updates s
    ON t.device_id = s.merge_key AND t.end_at IS NULL
    
    /* 1. If we matched the merge_key and the hash is different, CLOSE update the old record */
    WHEN MATCHED AND hash(s.device_type, s.brand, s.model, s.os, s.owner_customer_id, s.status) <> 
                     hash(t.device_type, t.brand, t.model, t.os, t.owner_customer_id, t.status)
    THEN UPDATE SET t.end_at = s.updated_at
    
    /* 2. If we didn't match (merge_key was NULL or ID is new), INSERT the new record */
    WHEN NOT MATCHED THEN
    INSERT (device_id, device_type, brand, model, os, owner_customer_id, status, updated_at, start_at, end_at)
    VALUES (s.device_id, s.device_type, s.brand, s.model, s.os, s.owner_customer_id, s.status, s.updated_at, s.updated_at, NULL)
""")